In [1]:
import gffpandas.gffpandas as gffpd
import pandas as pd
import numpy as np

### Working with Control pH 5 & 7, ArsS deletion pH 5 & 7

In [3]:
ctrl_ph7 = gffpd.read_gff3('../sample1_modification.gff') #Wild Type pH 7
arss_ph7 = gffpd.read_gff3('../sample2_modification.gff') #ArsS deletion pH 7, cannot detect pH change
ctrl_ph5 = gffpd.read_gff3('../sample5_modification.gff') #wild type pH 5
arss_ph5 = gffpd.read_gff3('../sample6_modification.gff') #ArsS deletion pH 5

#NOTE ANNOT, COUNTS, AND FREQ ARE ALL DATA FRAMES, NOT GFF DATA FRAMES
annot = pd.read_csv("../hpy_annot.tsv",sep='\t')#hpy annotation file
annot = annot.drop(columns=["seqname", "source", "feature", "score", "frame", "Prot_Size(est)", "attribute"]) #clean
tss = pd.read_csv("../TSS.csv")

In [4]:
# reading in counts
prom = pd.read_csv('promoter_methyl_counts.csv')
cdr = pd.read_csv('CDR_counts.csv')

In [5]:
ctrl_ph7 = ctrl_ph7.attributes_to_columns()
ctrl_ph5 = ctrl_ph5.attributes_to_columns()

arss_ph7 = arss_ph7.attributes_to_columns()
arss_ph5 = arss_ph5.attributes_to_columns()

ctrl_ph7 = ctrl_ph7[ctrl_ph7['IPDRatio'] >= '2']
ctrl_ph5 = ctrl_ph5[ctrl_ph5['IPDRatio'] >= '2']

arss_ph7 = arss_ph7[arss_ph7['IPDRatio'] >= '2']
arss_ph5 = arss_ph5[arss_ph5['IPDRatio'] >= '2']

In [6]:
# merging samples
frames = [ctrl_ph7, ctrl_ph5, arss_ph7, arss_ph5]
keys = ['ctrl pH7', 'ctrl pH5', 'arsS pH7', 'arsS pH5']
merged = pd.concat(frames, keys = keys)
merged = merged.reset_index(level=0).rename(columns={'level_0': 'sample'})
merged

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,4.29,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,4.46,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC
6,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD
12,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,5.78,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82780,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667645,1667645,630,-,.,context=ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG...,5.54,ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAGCGT,435,GATC,663,GATC
82782,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667648,1667648,470,-,.,context=ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTG...,4.78,ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG,418,DGAAGG,447,DGAAGG
82783,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667666,1667666,477,+,.,context=CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTT...,4.39,CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTTGGT,433,RCGDAD,451,RCGDAD
82784,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667731,1667731,643,-,.,context=TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGA...,5.19,TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGAATG,431,GAAGA/TCTTC,644,GAAGA


#### Gene Counts - Coding Region

In [7]:
counts = pd.DataFrame(columns=['gene', '1\' pH7','1\' pH5', '16\' pH7', '16\' pH5'])
counts['gene'] = annot['HP_number']
counts.head()

,gene,1' pH7,1' pH5,16' pH7,16' pH5
0,HP0001,NaN,NaN,NaN,NaN
1,HP0002,NaN,NaN,NaN,NaN
2,HP0003,NaN,NaN,NaN,NaN
3,HP0004,NaN,NaN,NaN,NaN
4,HP0005,NaN,NaN,NaN,NaN


In [8]:
for ind in counts.index: 
    counts.loc[ind, '1\' pH7']=((ctrl_ph7['start'].between(annot.loc[ind,'start'], annot.loc[ind,'end']+1))).sum()
    counts.loc[ind, '1\' pH5']=((ctrl_ph5['start'].between(annot.loc[ind,'start'], annot.loc[ind,'end']+1))).sum()
    counts.loc[ind, '16\' pH7']=((arss_ph7['start'].between(annot.loc[ind,'start'], annot.loc[ind,'end']+1))).sum()
    counts.loc[ind, '16\' pH5']=((arss_ph5['start'].between(annot.loc[ind,'start'], annot.loc[ind,'end']+1))).sum()

In [9]:
# if you want frequency
for ind in counts.index:
    gene_length = annot.loc[ind,'end'] - annot.loc[ind,'start']
    counts.loc[ind, '1\' pH7 density'] = counts.loc[ind, '1\' pH7'] / gene_length
    counts.loc[ind, '16\' pH7 density'] = counts.loc[ind, '16\' pH7'] / gene_length
    counts.loc[ind, '1\' pH5 density'] = counts.loc[ind, '1\' pH5'] / gene_length
    counts.loc[ind, '16\' pH5 density'] = counts.loc[ind, '16\' pH5'] / gene_length

In [10]:
counts.to_csv('CDR_counts.csv', index = False)

#### Gene Counts - Promoter Region 
Using Primary Promoters only

In [11]:
tss = pd.read_csv("../TSS.csv")

In [12]:
primary = tss[(tss['Primary'] == 1)]
primary

,TSS,Strand,Locus_tag,Name,Primary,Secondary,Internal,Antisense,Enriched,Location,Putative sRNA,Putative asRNA,Sequence -50 nt upstream + TSS (51nt),Comment
1,2081,-,HP0003,2-dehydro-3-deoxyphosphooctonate aldolase,1,0,0,0,1,0,0,0,GTTTGAATGCGCGCTTGCAACTCAACAACCTCTTAAGCTATGATTT...,-
3,2646,-,HP0004,carbonic anhydrase (icfA),1,0,0,0,1,0,0,0,TAACGCCATTATAACAAAAAGAAATGCAAGAATTTTAGCTATGATA...,-
6,2672,+,HP0005,orotidine 5'-phosphate decarboxylase,1,0,0,0,0,1,0,0,GATTAAAATTCGTTTGTTTTTAAAACGCTTATCATAGCTAAAATTC...,-
8,7321,-,HP0009,outer membrane protein,1,0,0,0,1,0,0,0,TTCTCCAAATGACAAAAAAAAAAAAAACGATTTTATGCTACAATGC...,-
9,9683,-,HP0011,co-chaperonin GroES,1,0,0,0,1,0,0,0,CTTTGTTTTTATGGCTTGACTTATCCCTAAAAATGCGCTATAGTTA...,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1873,1245780,-,sRNA_Li_15,sRNA_Li_15,1,0,0,0,1,0,1,0,AAACAACAACTTTACACCCCAAAAATGAAATTTTGAGGTATAATGT...,-
1878,497713,-,sRNA_Li_22,sRNA_Li_22,1,0,0,0,1,0,1,0,ATCAAGCGCTCGTTATGCCCTTGTTAGAAAAGTTTAAGCTAGAATT...,-
1891,1170409,-,sRNA_Li_42,sRNA_Li_42,1,0,0,0,1,0,0,1,TTTTTTCAAACAATTCTTCTTTGTTAAGGTAGCTAGTGATAATATA...,-
1896,1233920,+,sRNA_Li_50,sRNA_Li_50,1,0,0,0,1,0,1,0,ACTTGAGCGGTAGCGTGGATCTTGCATATTATGGCGTGTATACTAA...,-


In [13]:
def calculate_adjusted_tss(row):
    if row['Strand'] == '+':
        return row['TSS'] - 50  # Subtract 50 bp for the + strand
    if row['Strand'] == '-':
        return row['TSS'] + 50  # Add 50 bp for the - strand
    else:
        return row['TSS']

# Use .loc to ensure proper assignment without triggering the warning
primary.loc[:, '+/- 50 TSS'] = primary.apply(calculate_adjusted_tss, axis=1)

# Re-assign only the relevant columns
primary = primary[['TSS', '+/- 50 TSS', 'Strand', 'Locus_tag', 'Name']]

/var/folders/kw/19vs2__d3bl_tx91fly_4sg40000gn/T/ipykernel_12129/1892960449.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  primary.loc[:, '+/- 50 TSS'] = primary.apply(calculate_adjusted_tss, axis=1)


In [14]:
primary

,TSS,+/- 50 TSS,Strand,Locus_tag,Name
1,2081,2131,-,HP0003,2-dehydro-3-deoxyphosphooctonate aldolase
3,2646,2696,-,HP0004,carbonic anhydrase (icfA)
6,2672,2622,+,HP0005,orotidine 5'-phosphate decarboxylase
8,7321,7371,-,HP0009,outer membrane protein
9,9683,9733,-,HP0011,co-chaperonin GroES
...,...,...,...,...,...
1873,1245780,1245830,-,sRNA_Li_15,sRNA_Li_15
1878,497713,497763,-,sRNA_Li_22,sRNA_Li_22
1891,1170409,1170459,-,sRNA_Li_42,sRNA_Li_42
1896,1233920,1233870,+,sRNA_Li_50,sRNA_Li_50


In [15]:
p_counts = pd.DataFrame(columns=['gene', '-Val', 'strand', '1\' pH7',\
                                 '1\' pH5', '16\' pH7','16\' pH5'])
p_counts['gene'] = primary['Locus_tag']
p_counts['strand']= primary['Strand']
p_counts['TSS'] = primary['TSS']
p_counts['-Val'] = primary['+/- 50 TSS']
p_counts

,gene,-Val,strand,1' pH7,1' pH5,16' pH7,16' pH5,TSS
1,HP0003,2131,-,NaN,NaN,NaN,NaN,2081
3,HP0004,2696,-,NaN,NaN,NaN,NaN,2646
6,HP0005,2622,+,NaN,NaN,NaN,NaN,2672
8,HP0009,7371,-,NaN,NaN,NaN,NaN,7321
9,HP0011,9733,-,NaN,NaN,NaN,NaN,9683
...,...,...,...,...,...,...,...,...
1873,sRNA_Li_15,1245830,-,NaN,NaN,NaN,NaN,1245780
1878,sRNA_Li_22,497763,-,NaN,NaN,NaN,NaN,497713
1891,sRNA_Li_42,1170459,-,NaN,NaN,NaN,NaN,1170409
1896,sRNA_Li_50,1233870,+,NaN,NaN,NaN,NaN,1233920


In [16]:
tss_df = primary # creating an alias to avoid lots of restarting

In [17]:
tss_df.loc[:, 'TSS'] = tss_df['TSS'].astype(int)
tss_df['-Val'] = tss_df['+/- 50 TSS'].astype(int)
tss_df.loc[:, '-Val'] = tss_df['+/- 50 TSS'].astype(int)

ctrl_ph7.loc[:, 'start'] = ctrl_ph7['start'].astype(int)
ctrl_ph5.loc[:, 'start'] = ctrl_ph5['start'].astype(int)
arss_ph7.loc[:, 'start'] = arss_ph7['start'].astype(int)
arss_ph5.loc[:, 'start'] = arss_ph5['start'].astype(int)

/var/folders/kw/19vs2__d3bl_tx91fly_4sg40000gn/T/ipykernel_12129/1314104552.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tss_df['-Val'] = tss_df['+/- 50 TSS'].astype(int)


In [18]:
for ind in p_counts.index:
    tss_value = int(tss_df.loc[ind, 'TSS'])
    val_value = int(tss_df.loc[ind, '-Val'])
    strand = tss_df.loc[ind, 'Strand']  # Get the strand information
    
    if strand == '+':
        #for + strand, the range is upstream (val_value) to the TSS
        p_counts.loc[ind, '1\' pH7'] = (ctrl_ph7['start'].between(val_value, tss_value + 1)).sum()
        p_counts.loc[ind, '1\' pH5'] = (ctrl_ph5['start'].between(val_value, tss_value + 1)).sum()
        p_counts.loc[ind, '16\' pH7'] = (arss_ph7['start'].between(val_value, tss_value + 1)).sum()
        p_counts.loc[ind, '16\' pH5'] = (arss_ph5['start'].between(val_value, tss_value + 1)).sum()
        
    if strand == '-':
        #for- strand, the range is from the TSS downstream (tss_value) to val_value
        p_counts.loc[ind, '1\' pH7'] = (ctrl_ph7['start'].between(tss_value, val_value + 1)).sum()
        p_counts.loc[ind, '1\' pH5'] = (ctrl_ph5['start'].between(tss_value, val_value + 1)).sum()
        p_counts.loc[ind, '16\' pH7'] = (arss_ph7['start'].between(tss_value, val_value + 1)).sum()
        p_counts.loc[ind, '16\' pH5'] = (arss_ph5['start'].between(tss_value, val_value + 1)).sum()

In [19]:
p_counts

,gene,-Val,strand,1' pH7,1' pH5,16' pH7,16' pH5,TSS
1,HP0003,2131,-,3,3,2,1,2081
3,HP0004,2696,-,0,0,0,0,2646
6,HP0005,2622,+,1,1,1,1,2672
8,HP0009,7371,-,0,0,0,0,7321
9,HP0011,9733,-,0,0,0,0,9683
...,...,...,...,...,...,...,...,...
1873,sRNA_Li_15,1245830,-,3,3,3,3,1245780
1878,sRNA_Li_22,497763,-,0,0,0,0,497713
1891,sRNA_Li_42,1170459,-,3,3,3,3,1170409
1896,sRNA_Li_50,1233870,+,2,2,2,2,1233920


In [20]:
p_counts.to_csv('promoter_methyl_counts.csv', index = False)

## Attach Genetic Region to Merged (Prom, CDR, intergenetic regions)
1. tackle the promoter definition region
2. since DNA is double stranded at the promoter, and not locally melted -- we can simply create
   an array of 0s & 1s for promoter regions
       Methylations on both strands will impact the ribosome binding
3. after taking 0s and 1s, index each start and compare if there's a 0 or 1 indicating presence of a promoter

In [21]:
merged

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,4.29,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,4.46,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC
6,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD
12,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,5.78,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82780,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667645,1667645,630,-,.,context=ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG...,5.54,ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAGCGT,435,GATC,663,GATC
82782,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667648,1667648,470,-,.,context=ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTG...,4.78,ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG,418,DGAAGG,447,DGAAGG
82783,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667666,1667666,477,+,.,context=CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTT...,4.39,CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTTGGT,433,RCGDAD,451,RCGDAD
82784,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667731,1667731,643,-,.,context=TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGA...,5.19,TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGAATG,431,GAAGA/TCTTC,644,GAAGA


In [22]:
# separate the plus and minus ends strands of promoters
prom_minus = prom[prom['strand'] == '-']
prom_plus = prom[prom['strand'] == '+']

In [23]:
prom_combined = pd.concat([prom_plus, prom_minus], ignore_index=True)

val_arr_plus = prom_plus['-Val'].to_numpy()  #for the + strand
tss_arr_plus = prom_plus['TSS'].to_numpy()

val_arr_minus = prom_minus['-Val'].to_numpy()  #for the - strand
tss_arr_minus = prom_minus['TSS'].to_numpy()

# Determine the final genome length (maximum TSS or -Val value + padding)
end = max(max(val_arr_plus), max(val_arr_minus), max(tss_arr_plus), max(tss_arr_minus))
final = end + 100  # Adjust for padding

# Create an array of zeros for the genome length
arr2 = np.zeros(final + 1, dtype=int)

# Combined loop for both + and - strand promoters
for index, row in prom_combined.iterrows():
    tss_value = row['TSS']
    val_value = row['-Val']
    strand = row['strand']  # Retrieve the strand information for each row

    if strand == '+':
        # For + strand, range is from -Val to TSS
        arr2[val_value:tss_value+1] = 1  # Mark positions between -Val and TSS with 1

    elif strand == '-':
        # For - strand, range is from TSS to -Val (reverse orientation)
        arr2[tss_value:val_value+1] = 1  # Mark positions between TSS and -Val with 1

# Output for checking specific regions and length
print(arr2[2000:2151])  # For example, between positions 2000 and 2150
print(len(arr2))        # Length of the array for validation

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0]
1667018


In [24]:
genome_array = arr2
methylation_df = merged

def classify_promoter(row):
    start_location = row['start']
    if start_location < len(genome_array):  # Check if start location is within range
        if genome_array[start_location] == 1:
            return 'Promoter'
        else:
            return np.nan
    else:
        return np.nan  # Return NaN if start location is out of range

# Apply the function to the DataFrame
methylation_df['Classification'] = methylation_df.apply(classify_promoter, axis=1)

In [25]:
methylation_df

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif,Classification
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,4.29,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG,NaN
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC,NaN
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,4.46,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC,NaN
6,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD,NaN
12,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,5.78,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82780,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667645,1667645,630,-,.,context=ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG...,5.54,ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAGCGT,435,GATC,663,GATC,NaN
82782,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667648,1667648,470,-,.,context=ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTG...,4.78,ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG,418,DGAAGG,447,DGAAGG,NaN
82783,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667666,1667666,477,+,.,context=CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTT...,4.39,CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTTGGT,433,RCGDAD,451,RCGDAD,NaN
82784,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667731,1667731,643,-,.,context=TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGA...,5.19,TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGAATG,431,GAAGA/TCTTC,644,GAAGA,NaN


In [26]:
# coding region df:
start_arr = annot['start'].to_numpy()
end_arr = annot['end'].to_numpy() 

genome_length = max(end_arr) + 1  # Add 1 to make the array 0-indexed and inclusive
genome_array = np.zeros(genome_length, dtype=int)  # Create an array of zeros of genome length

for index, row in annot.iterrows():
    start = row['start']
    end = row['end']
    genome_array[start:end+1] = 1  # Inclusive of the 'end' position

print(genome_array[2000:2151])  # For example, between positions 2000 and 2150
print(len(genome_array))  

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1]
1667801


In [27]:
def classify_coding_region(row):
    start_location = row['start']
    if start_location < len(genome_array):
        if genome_array[start_location] == 1:
            return 'Coding Region'
    else:
        return np.nan

methylation_df['Classification'] = methylation_df.apply(classify_coding_region, axis=1)

In [28]:
## applying all together

import numpy as np
import pandas as pd

# Combine promoter DataFrames, + and -
prom_minus = prom[prom['strand'] == '-']
prom_plus = prom[prom['strand'] == '+']

prom_combined = pd.concat([prom_plus, prom_minus], ignore_index=True)

# Extract the necessary arrays for + and - strand promoters
val_arr_plus = prom_plus['-Val'].to_numpy()
tss_arr_plus = prom_plus['TSS'].to_numpy()

val_arr_minus = prom_minus['-Val'].to_numpy()
tss_arr_minus = prom_minus['TSS'].to_numpy()

# Coding region arrays
start_arr = annot['start'].to_numpy()
end_arr = annot['end'].to_numpy()

# Determine the final genome length based on both promoter and coding region data
end_promoters = max(max(val_arr_plus), max(val_arr_minus), max(tss_arr_plus), max(tss_arr_minus))
end_coding = max(end_arr)
final_genome_length = max(end_promoters, end_coding) + 100  # Adding padding

# Create an array initialized with zeros (intergenic regions)
genome_array = np.zeros(final_genome_length + 1, dtype=int)

# First pass: Mark promoter regions as 1
for index, row in prom_combined.iterrows():
    tss_value = row['TSS']
    val_value = row['-Val']
    strand = row['strand']  # Get strand information
    
    if strand == '+':
        # Mark positions from -Val to TSS for + strand as promoter
        genome_array[val_value:tss_value+1] = 1  # Set promoter region to 1
    elif strand == '-':
        # Mark positions from TSS to -Val for - strand as promoter
        genome_array[tss_value:val_value+1] = 1  # Set promoter region to 1

# Second pass: Mark coding regions as 2, or as 3 if they overlap with promoters
for index, row in annot.iterrows():
    start = row['start']
    end = row['end']
    
    # If the region overlaps with a promoter (1), mark it as both (3)
    # Otherwise, mark it as a coding region (2)
    for pos in range(start, end+1):
        if genome_array[pos] == 1:
            genome_array[pos] = 3  # Both promoter and coding region
        elif genome_array[pos] == 0:
            genome_array[pos] = 2  # Mark as coding region

# Output for checking specific regions and length
print(genome_array[2000:2151])  # For example, between positions 2000 and 2150
print(len(genome_array))        # Length of the array for validation

[2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2]
1667901


In [29]:
def classify_coding_region(row):
    start_location = row['start']
    if start_location < len(genome_array):
        if genome_array[start_location] == 1:
            return 'Promoter Region'
        elif genome_array[start_location] == 2:
            return 'Coding Region'
        elif genome_array[start_location] == 3:
            return 'Promoter and Coding Region'
        else:
            return 'Intergenic Region'
    else:
        return np.nan

methylation_df['Classification'] = methylation_df.apply(classify_coding_region, axis=1)

In [30]:
methylation_df[methylation_df['Classification'] == 'Promoter and Coding Region']

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif,Classification
113,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2101,2101,621,+,.,context=GAATGAAATCATAGCTTAAGAGGTTGTTGAGTTGCAAG...,4.72,GAATGAAATCATAGCTTAAGAGGTTGTTGAGTTGCAAGCGC,556,GAGG,587,GAGG,Promoter and Coding Region
115,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2117,2117,137,+,.,coverage=559;context=TAAGAGGTTGTTGAGTTGCAAGCGC...,2.22,TAAGAGGTTGTTGAGTTGCAAGCGCGCATTCAAACGCTCTG,559,None,85,None,Promoter and Coding Region
118,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2128,2128,135,+,.,context=TGAGTTGCAAGCGCGCATTCAAACGCTCTGTAAGCCAT...,2.11,TGAGTTGCAAGCGCGCATTCAAACGCTCTGTAAGCCATGAA,527,HHTCAVNNNNNNTGY,85,HHTCAVNNNNNNTGY,Promoter and Coding Region
2245,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,43167,43167,537,+,.,context=TAAAAGAGTTGGTTAAGGACATGTTAGAATACGATTTA...,4.35,TAAAAGAGTTGGTTAAGGACATGTTAGAATACGATTTAAAA,529,CATG,511,CATG,Promoter and Coding Region
2246,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,43168,43168,651,-,.,context=CTTTTAAATCGTATTCTAACATGTCCTTAACCAACTCT...,4.54,CTTTTAAATCGTATTCTAACATGTCCTTAACCAACTCTTTT,549,CATG,657,CATG,Promoter and Coding Region
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82018,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1651561,1651561,552,-,.,context=TTACAGGAAGAACTTCAATCATGGTGAAAAAACGCATG...,5.44,TTACAGGAAGAACTTCAATCATGGTGAAAAAACGCATGGAG,412,CATG,529,CATG,Promoter and Coding Region
82019,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1651571,1651571,523,-,.,context=ATCGCTATCATTACAGGAAGAACTTCAATCATGGTGAA...,5.32,ATCGCTATCATTACAGGAAGAACTTCAATCATGGTGAAAAA,419,GAAGA/TCTTC,502,GAAGA,Promoter and Coding Region
82020,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m4C,1651572,1651572,371,+,.,context=TTTTCACCATGATTGAAGTTCTTCCTGTAATGATAGCG...,3.40,TTTTCACCATGATTGAAGTTCTTCCTGTAATGATAGCGATT,409,GAAGA/TCTTC,340,TCTTC,Promoter and Coding Region
82188,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1654617,1654617,701,+,.,context=TCGTTTTCATAGTCCTTGTGATCCAAATGGCAATGCGT...,6.05,TCGTTTTCATAGTCCTTGTGATCCAAATGGCAATGCGTGTC,435,GATC,673,GATC,Promoter and Coding Region


## Attach HP numbers and genetic category

In [39]:
from tqdm.auto import tqdm
tqdm.pandas()

In [31]:
methylation_df.head()

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif,Classification
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,4.29,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG,Intergenic Region
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC,Coding Region
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,4.46,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC,Coding Region
6,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD,Coding Region
12,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,5.78,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD,Coding Region


In [ ]:
def find_promoter_id(start_val, p_counts):
    match = p_counts[((p_counts['-Val'] <= start_val) & (p_counts['TSS'] >= start_val) & (p_counts['strand'] == '+')) |
                     ((p_counts['TSS'] <= start_val) & (p_counts['-Val'] >= start_val) & (p_counts['strand'] == '-'))]
    # Return the first matching gene, or None if no match
    if not match.empty:
        return match.iloc[0]['gene']  # You can also return other columns like 'Locus_tag'
    return None

/var/folders/kw/19vs2__d3bl_tx91fly_4sg40000gn/T/ipykernel_12129/1504734440.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['gene_ID'] = sub['start'].apply(find_gene_id, p_counts=p_counts)


,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif,Classification,gene_ID
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,4.29,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG,Intergenic Region,None
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC,Coding Region,None
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,4.46,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC,Coding Region,None
6,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD,Coding Region,None
12,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,5.78,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD,Coding Region,None


In [41]:
methylation_df['gene_ID'] = methylation_df['start'].progress_apply(find_gene_id, p_counts=p_counts)
methylation_df[methylation_df['gene_ID'].notna()]

  0%|          | 0/237130 [00:00<?, ?it/s]

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif,Classification,gene_ID
113,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2101,2101,621,+,.,context=GAATGAAATCATAGCTTAAGAGGTTGTTGAGTTGCAAG...,4.72,GAATGAAATCATAGCTTAAGAGGTTGTTGAGTTGCAAGCGC,556,GAGG,587,GAGG,Promoter and Coding Region,HP0003
115,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2117,2117,137,+,.,coverage=559;context=TAAGAGGTTGTTGAGTTGCAAGCGC...,2.22,TAAGAGGTTGTTGAGTTGCAAGCGCGCATTCAAACGCTCTG,559,None,85,None,Promoter and Coding Region,HP0003
118,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2128,2128,135,+,.,context=TGAGTTGCAAGCGCGCATTCAAACGCTCTGTAAGCCAT...,2.11,TGAGTTGCAAGCGCGCATTCAAACGCTCTGTAAGCCATGAA,527,HHTCAVNNNNNNTGY,85,HHTCAVNNNNNNTGY,Promoter and Coding Region,HP0003
146,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,2630,2630,673,-,.,context=AGCGTTTTAAAAACAAACGAATTTTAATCAAAATGAGA...,5.42,AGCGTTTTAAAAACAAACGAATTTTAATCAAAATGAGATTT,535,RCGDAD,685,RCGDAD,Promoter Region,HP0005
675,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,12653,12653,390,+,.,context=CGCTTGAAAGGGAGTTTTTGAGGGTTTTAGGGGTTTTC...,3.37,CGCTTGAAAGGGAGTTTTTGAGGGTTTTAGGGGTTTTCTTT,436,GAGG,360,GAGG,Promoter Region,HP0014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82190,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1654645,1654645,463,+,.,context=GGCAATGCGTGTCTATAAACATGCTTTTATCCTATGGT...,4.05,GGCAATGCGTGTCTATAAACATGCTTTTATCCTATGGTTTT,434,CATG,412,CATG,Promoter Region,HP1574
82191,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1654646,1654646,367,-,.,context=TAAAACCATAGGATAAAAGCATGTTTATAGACACGCAT...,3.55,TAAAACCATAGGATAAAAGCATGTTTATAGACACGCATTGC,422,CATG,352,CATG,Promoter Region,HP1574
82476,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1660580,1660580,485,+,.,context=TAAAATCTTATAGGTGTAAGAGGCGGGTTTTATGTTAC...,4.51,TAAAATCTTATAGGTGTAAGAGGCGGGTTTTATGTTACAAT,436,GAGG,471,GAGG,Promoter Region,HP1582
82604,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1663527,1663527,512,+,.,context=GAGGCAAGAGCTTAAAAAAGAGGGTGTTAAAAAAGCGC...,4.65,GAGGCAAGAGCTTAAAAAAGAGGGTGTTAAAAAAGCGCGTT,428,GAGG,524,GAGG,Promoter Region,HP1585


In [48]:
methylation_df.rename(columns={'gene_ID': 'promoter_ID'}, inplace=True)

In [60]:
def find_coding_reg_id(start_val, p_counts):
    match = annot[((annot['start'] <= start_val) & (annot['end'] >= start_val))]
    # Return the first matching gene, or None if no match
    if not match.empty:
        return match.iloc[0]['HP_number']  # You can also return other columns like 'Locus_tag'
    return None

# Apply the function to the 'start' column in sub
methylation_df['CodingRegion_ID'] = methylation_df['start'].progress_apply(find_coding_reg_id, p_counts=p_counts)


  0%|          | 0/237130 [00:00<?, ?it/s]

In [64]:
methylation_df.to_csv("methylation_df.csv")

## Adding genetic function to DF

In [69]:
gene_categories = pd.read_csv('hpy_annot - hpy_annot.csv')
gene_categories

,seqname,source,feature,HP_number,Unnamed: 4,start,end,Putative_Id,Orf_Size,Prot_Size(est),score,strand,frame
0,.,forsyth,gene,HP0001,TRANSCRIPTION,220,633,transcription termination factor NusB (nusB) {...,413,15180.0,.,-,.
1,.,forsyth,gene,HP0002,"BIOSYNTHESIS OF COFACTORS, PROSTHETIC GROUPS, ...",638,1105,riboflavin synthase beta chain (ribE) {Haemoph...,467,17160.0,.,-,.
2,.,forsyth,gene,HP0003,CELL ENVELOPE,1118,1945,3-deoxy-d-manno-octulosonic acid 8-phosphate s...,827,30360.0,.,-,.
3,.,forsyth,gene,HP0004,CENTRAL INTERMEDIARY METABOLISM,1935,2597,carbonic anhydrase (icfA) {Synechococcus sp.},662,24310.0,.,-,.
4,.,forsyth,gene,HP0005,"PURINES, PYRIMIDINES, NUCLEOSIDES AND NUCLEOTIDES",2719,3399,orotidine 5'-phosphate decarboxylase (pyrF) {B...,680,24970.0,.,+,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1585,.,forsyth,gene,HP1586,HYPOTHETICAL PROTEIEN,1664450,1664785,hypothetical protein,335,12320.0,.,+,.
1586,.,forsyth,gene,HP1587,HYPOTHETICAL PROTEIEN,1664880,1665344,conserved hypothetical protein {Escherichia coli},464,17050.0,.,-,.
1587,.,forsyth,gene,HP1588,HYPOTHETICAL PROTEIEN,1666032,1666790,conserved hypothetical protein {Escherichia coli},758,27830.0,.,-,.
1588,.,forsyth,gene,HP1589,HYPOTHETICAL PROTEIEN,1667060,1667680,conserved hypothetical protein {Escherichia coli},620,22770.0,.,-,.


In [70]:
gene_categories.rename(columns={'Unnamed: 4': 'gene_type'}, inplace=True)

In [71]:
gene_categories

,seqname,source,feature,HP_number,gene_type,start,end,Putative_Id,Orf_Size,Prot_Size(est),score,strand,frame
0,.,forsyth,gene,HP0001,TRANSCRIPTION,220,633,transcription termination factor NusB (nusB) {...,413,15180.0,.,-,.
1,.,forsyth,gene,HP0002,"BIOSYNTHESIS OF COFACTORS, PROSTHETIC GROUPS, ...",638,1105,riboflavin synthase beta chain (ribE) {Haemoph...,467,17160.0,.,-,.
2,.,forsyth,gene,HP0003,CELL ENVELOPE,1118,1945,3-deoxy-d-manno-octulosonic acid 8-phosphate s...,827,30360.0,.,-,.
3,.,forsyth,gene,HP0004,CENTRAL INTERMEDIARY METABOLISM,1935,2597,carbonic anhydrase (icfA) {Synechococcus sp.},662,24310.0,.,-,.
4,.,forsyth,gene,HP0005,"PURINES, PYRIMIDINES, NUCLEOSIDES AND NUCLEOTIDES",2719,3399,orotidine 5'-phosphate decarboxylase (pyrF) {B...,680,24970.0,.,+,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1585,.,forsyth,gene,HP1586,HYPOTHETICAL PROTEIEN,1664450,1664785,hypothetical protein,335,12320.0,.,+,.
1586,.,forsyth,gene,HP1587,HYPOTHETICAL PROTEIEN,1664880,1665344,conserved hypothetical protein {Escherichia coli},464,17050.0,.,-,.
1587,.,forsyth,gene,HP1588,HYPOTHETICAL PROTEIEN,1666032,1666790,conserved hypothetical protein {Escherichia coli},758,27830.0,.,-,.
1588,.,forsyth,gene,HP1589,HYPOTHETICAL PROTEIEN,1667060,1667680,conserved hypothetical protein {Escherichia coli},620,22770.0,.,-,.


In [72]:
methylation_df

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif,Classification,promoter_ID,CodingRegion_ID
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,4.29,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG,Intergenic Region,None,None
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC,Coding Region,None,HP0001
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,4.46,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC,Coding Region,None,HP0001
6,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD,Coding Region,None,HP0001
12,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,5.78,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD,Coding Region,None,HP0001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82780,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667645,1667645,630,-,.,context=ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG...,5.54,ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAGCGT,435,GATC,663,GATC,Coding Region,None,HP1589
82782,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667648,1667648,470,-,.,context=ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTG...,4.78,ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG,418,DGAAGG,447,DGAAGG,Coding Region,None,HP1589
82783,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667666,1667666,477,+,.,context=CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTT...,4.39,CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTTGGT,433,RCGDAD,451,RCGDAD,Coding Region,None,HP1589
82784,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667731,1667731,643,-,.,context=TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGA...,5.19,TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGAATG,431,GAAGA/TCTTC,644,GAAGA,Coding Region,None,HP1590


In [79]:
methylation_df = pd.merge(methylation_df, 
                          gene_categories[['HP_number', 'gene_type']], 
                          left_on='CodingRegion_ID', 
                          right_on='HP_number', 
                          how='left')

methylation_df.rename(columns={'gene_type': 'CodingRegion_gene_type'}, inplace=True)

methylation_df = pd.merge(methylation_df, 
                          gene_categories[['HP_number', 'gene_type']], 
                          left_on='promoter_ID', 
                          right_on='HP_number', 
                          how='left')

methylation_df.rename(columns={'gene_type': 'Promoter_gene_type'}, inplace=True)

methylation_df.drop(columns=['HP_number_x', 'HP_number_y'], inplace=True, errors='ignore')


In [88]:
methylation_df = methylation_df.loc[:, ~methylation_df.columns.duplicated()]
methylation_df['Promoter_gene_type'] = methylation_df['Promoter_gene_type'].astype(str)
methylation_df['Promoter_gene_type'] = methylation_df['Promoter_gene_type'].str.replace(r'\r\n', ' ', regex=True)

print(methylation_df['Promoter_gene_type'].value_counts())

Promoter_gene_type
nan                                                           234413
HYPOTHETICAL PROTEIEN                                           1111
CELL ENVELOPE                                                    309
CELLULAR PROCESSES                                               235
TRANSPORT AND BINDING PROTEINS                                   195
REPLICATION                                                      188
TRANSLATION                                                      148
ENERGY METABOLISM                                                128
BIOSYNTHESIS OF COFACTORS, PROSTHETIC GROUPS, AND CARRIERS       126
AMINO-ACID BIOSYNTHESIS                                           96
PURINES, PYRIMIDINES, NUCLEOSIDES AND NUCLEOTIDES                 81
OTHER CATEGORIES                                                  44
CENTRAL INTERMEDIARY METABOLISM                                   20
TRANSCRIPTION                                                     16
FATTY ACID AND 

In [91]:
methylation_df['CodingRegion_gene_type'] = methylation_df['CodingRegion_gene_type'].astype(str)
methylation_df['CodingRegion_gene_type'] = methylation_df['CodingRegion_gene_type'].str.replace(r'\r\n', ' ', regex=True)

methylation_df['CodingRegion_gene_type'] = methylation_df['CodingRegion_gene_type'].replace({
    'CELLULAR PROCESSE': 'CELLULAR PROCESSES',
    'HYPOTHETICAL PROTEIEN': 'HYPOTHETICAL PROTEIN'
})
print(methylation_df['CodingRegion_gene_type'].value_counts())

CodingRegion_gene_type
HYPOTHETICAL PROTEIN                                          72656
CELL ENVELOPE                                                 26539
REPLICATION                                                   17431
TRANSLATION                                                   17169
ENERGY METABOLISM                                             16757
CELLULAR PROCESSES                                            16465
nan                                                           16255
TRANSPORT AND BINDING PROTEINS                                15110
BIOSYNTHESIS OF COFACTORS, PROSTHETIC GROUPS, AND CARRIERS     7220
AMINO-ACID BIOSYNTHESIS                                        6822
PURINES, PYRIMIDINES, NUCLEOSIDES AND NUCLEOTIDES              5956
CENTRAL INTERMEDIARY METABOLISM                                4268
REGULATORY FUNCTIONS                                           4228
OTHER CATEGORIES                                               3946
FATTY ACID AND PHOSPHOLIP

In [92]:
methylation_df

,sample,seq_id,source,type,start,end,score,strand,phase,attributes,...,context,coverage,id,identificationQv,motif,Classification,promoter_ID,CodingRegion_ID,CodingRegion_gene_type,Promoter_gene_type
0,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,217,217,579,-,.,context=ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAA...,...,ACTCAAAAACCCTTGAATTGAGGGTGTTTTATACCTAAATT,547,GAGG,535,GAGG,Intergenic Region,None,None,nan,nan
1,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,...,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC,Coding Region,None,HP0001,TRANSCRIPTION,nan
2,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,257,257,604,-,.,context=ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGC...,...,ATTTTTAAACGCTATCTTGGATTCTTTGAGCAAAAAGCTCA,548,GANTC,597,GANTC,Coding Region,None,HP0001,TRANSCRIPTION,nan
3,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,...,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD,Coding Region,None,HP0001,TRANSCRIPTION,nan
4,ctrl pH7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,368,368,769,-,.,context=GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGC...,...,GATTTTACGCTTAGGAGCGTATGAAATTGGCTTCACGCCCA,562,RCGDAD,738,RCGDAD,Coding Region,None,HP0001,TRANSCRIPTION,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237125,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667645,1667645,630,-,.,context=ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG...,...,ACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAGCGT,435,GATC,663,GATC,Coding Region,None,HP1589,HYPOTHETICAL PROTEIN,nan
237126,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667648,1667648,470,-,.,context=ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTG...,...,ATTACGCCAAGTACCCAAGAAGGATCGCTGAAGAATTGCAG,418,DGAAGG,447,DGAAGG,Coding Region,None,HP1589,HYPOTHETICAL PROTEIN,nan
237127,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667666,1667666,477,+,.,context=CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTT...,...,CCTTCTTGGGTACTTGGCGTAATCATAGCCATACCTTTGGT,433,RCGDAD,451,RCGDAD,Coding Region,None,HP1589,HYPOTHETICAL PROTEIN,nan
237128,arsS pH5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,1667731,1667731,643,-,.,context=TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGA...,...,TGCGCTTGTTTATGATGAAGATGGCACACTAAGAATGAATG,431,GAAGA/TCTTC,644,GAAGA,Coding Region,None,HP1590,HYPOTHETICAL PROTEIN,nan


In [93]:
methylation_df.to_csv("methylation_df.csv")